In [ ]:
import os
import re
import numpy as np
import pandas as pd
import soundfile as sf
from scipy.ndimage import gaussian_filter1d

COMPETITION_DIR = '/kaggle/input/competitions/birdclef-2026'
TAXONOMY_PATH = os.path.join(COMPETITION_DIR, 'taxonomy.csv')
TEST_SOUNDSCAPE_DIR = os.path.join(COMPETITION_DIR, 'test_soundscapes')
TRAIN_LABELS_PATH = os.path.join(COMPETITION_DIR, 'train_soundscapes_labels.csv')

CHUNK_SECONDS = 5
TIME_BIN_HOURS = 2    # bin size in hours -> 12 bins per day
TIME_SIGMA = 1.0      # Gaussian smoothing bandwidth in time bins (circular)
MONTH_SIGMA = 1.0     # Gaussian smoothing bandwidth in months (circular)

In [ ]:
taxonomy = pd.read_csv(TAXONOMY_PATH)
class_labels = sorted(taxonomy['primary_label'].astype(str).tolist())
n_classes = len(class_labels)
print(f'{n_classes} classes from taxonomy')

df = pd.read_csv(TRAIN_LABELS_PATH)

def parse_filename(fn):
    m = re.match(r'.*_(S\d+)_(\d{8})_(\d{6})\.ogg', fn)
    if m:
        site = m.group(1)
        month = int(m.group(2)[4:6])
        hour = int(m.group(3)[:2])
        time_bin = hour // TIME_BIN_HOURS
        return site, time_bin, month
    return None, None, None

df['site'], df['time_bin'], df['month'] = zip(*df['filename'].map(parse_filename))

# Explode multi-label and restrict to taxonomy classes
df_exp = df.copy()
df_exp['label'] = df_exp['primary_label'].str.split(';')
df_exp = df_exp.explode('label')
df_exp = df_exp[df_exp['label'].isin(class_labels)]
print(f'{df_exp["label"].nunique()} training classes found in taxonomy')

In [ ]:
# --- Global prior P(class) ---
global_counts = df_exp['label'].value_counts()
global_prior = pd.Series(0.0, index=class_labels)
global_prior.update(global_counts)
global_prior = (global_prior + 1) / (global_prior + 1).sum()  # Laplace smoothing
print('Global prior computed')

# --- P(class | site) ---
site_label_counts = df_exp.groupby(['site', 'label']).size().unstack(fill_value=0)
site_label_counts = site_label_counts.reindex(columns=class_labels, fill_value=0) + 1  # Laplace
p_class_given_site = site_label_counts.div(site_label_counts.sum(axis=1), axis=0)
known_sites = set(p_class_given_site.index)
print(f'P(class|site) computed for sites: {sorted(known_sites)}')

def smooth_circular(counts_df, n_bins, sigma):
    """Reindex to n_bins, tile 3x, Gaussian smooth, take middle, Laplace smooth, normalise."""
    arr = counts_df.reindex(index=range(n_bins), columns=class_labels, fill_value=0).astype(float).values
    arr_tiled = np.tile(arr, (3, 1))
    arr_smoothed = gaussian_filter1d(arr_tiled, sigma=sigma, axis=0)
    arr_smoothed = arr_smoothed[n_bins:2*n_bins]
    arr_smoothed += 1  # Laplace smoothing
    df_smoothed = pd.DataFrame(arr_smoothed, index=range(n_bins), columns=class_labels)
    return df_smoothed.div(df_smoothed.sum(axis=1), axis=0)

# --- P(class | time bin) ---
n_time_bins = 24 // TIME_BIN_HOURS
time_label_counts = df_exp.groupby(['time_bin', 'label']).size().unstack(fill_value=0)
p_class_given_time = smooth_circular(time_label_counts, n_bins=n_time_bins, sigma=TIME_SIGMA)
print(f'P(class|time) computed ({n_time_bins} bins)')

# --- P(class | month) --- months are 1-indexed, map to 0-indexed for smoothing
month_label_counts = df_exp.groupby(['month', 'label']).size().unstack(fill_value=0)
month_label_counts.index = month_label_counts.index - 1
p_class_given_month = smooth_circular(month_label_counts, n_bins=12, sigma=MONTH_SIGMA)
print('P(class|month) computed')

In [ ]:
LOG_EPS = 1e-10

log_global = np.log(global_prior.values + LOG_EPS)
log_p_time = np.log(p_class_given_time.values + LOG_EPS)   # shape (n_time_bins, n_classes)
log_p_month = np.log(p_class_given_month.values + LOG_EPS)  # shape (12, n_classes)
log_p_site = {
    site: np.log(p_class_given_site.loc[site].values + LOG_EPS)
    for site in known_sites
}

def get_scores(site, time_bin, month):
    """Naive Bayes: log P(c|s,t,m) ∝ log P(c|s) + log P(c|t) + log P(c|m) - 2*log P(c).
    Falls back gracefully for unknown/missing values."""
    terms = []
    n_terms = 0
    if site in log_p_site:
        terms.append(log_p_site[site])
        n_terms += 1
    if time_bin is not None:
        terms.append(log_p_time[time_bin])
        n_terms += 1
    if month is not None:
        terms.append(log_p_month[month - 1])
        n_terms += 1

    if n_terms == 0:
        log_scores = log_global
    else:
        log_scores = sum(terms) - (n_terms - 1) * log_global

    log_scores = log_scores - log_scores.max()
    scores = np.exp(log_scores)
    scores /= scores.sum()
    return scores

print('Combination function defined')

In [ ]:
test_soundscapes = sorted(
    f for f in os.listdir(TEST_SOUNDSCAPE_DIR) if f.endswith('.ogg')
)
print(f'{len(test_soundscapes)} test soundscapes')

rows = []
for filename in test_soundscapes:
    soundscape_id = filename.split('.')[0]
    site, time_bin, month = parse_filename(filename)

    if site not in known_sites:
        print(f'  Unknown site: {site} in {filename}')
        site = None

    scores = get_scores(site, time_bin, month)

    filepath = os.path.join(TEST_SOUNDSCAPE_DIR, filename)
    info = sf.info(filepath)
    n_chunks = int(np.ceil(info.duration / CHUNK_SECONDS))

    for chunk_idx in range(n_chunks):
        end_time = (chunk_idx + 1) * CHUNK_SECONDS
        row_id = f'{soundscape_id}_{end_time}'
        rows.append([row_id] + scores.tolist())

submission = pd.DataFrame(rows, columns=['row_id'] + class_labels)
submission.to_csv('submission.csv', index=False)
print(f'submission.csv: {len(submission)} rows x {len(submission.columns)} columns')
submission.head()